[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jdsanch1/simrc/blob/master/01.%20Parte%201/05.%20Clase%205/05Class%20NB.ipynb)
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jdsanch1/SimRC/master?labpath=01.%20Parte%201%2F05.%20Clase%205%2F05Class%20NB.ipynb)

In [ ]:
import importlib, subprocess, sys

_required = ["yfinance", "pandas", "numpy", "matplotlib", "seaborn", "scipy", "sklearn", "statsmodels", "cvxpy"]
_missing  = [pkg for pkg in _required if importlib.util.find_spec(pkg) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + _missing)

# Clase 5: Covarianza robusta — Shrunk Covariance

**[Juan Diego Sánchez Torres](https://www.researchgate.net/profile/Juan_Diego_Sanchez_Torres)**  
*Profesor*, [MAF ITESO](http://maf.iteso.mx/web/general/detalle?group_id=5858156)  
dsanchez@iteso.mx

## Objetivos de aprendizaje

- Identificar las limitaciones de la **covarianza muestral** (inestabilidad, sobreajuste).
- Aplicar estimadores de **contracción** (shrinkage): `ShrunkCovariance` y `LedoitWolf`.
- Comparar matrices de covarianza (heatmaps, eigenvalores, número de condición).
- Evaluar el impacto de la contracción en la **frontera eficiente** y los pesos óptimos.
- Seleccionar activos con clustering antes de optimizar.

---

## Introducción teórica

### El problema de estimar la covarianza

En la Clase 4 usamos la **covarianza muestral** $\hat{\boldsymbol{\Sigma}}$ para optimizar portafolios. Sin embargo, esta estimación tiene problemas serios cuando el número de activos $n$ es grande relativo al número de observaciones $T$:

1. **Inestabilidad**: La covarianza muestral tiene $n(n+1)/2$ parámetros. Con $n = 50$ activos, son 1,275 parámetros a estimar.
2. **Error de estimación amplificado**: Pequeños errores en $\hat{\boldsymbol{\Sigma}}$ producen pesos óptimos muy diferentes (Michaud, 1989).
3. **Portafolios extremos**: El optimizador "explota" los errores de estimación, generando posiciones grandes y poco diversificadas.

### Solución: estimadores de contracción (shrinkage)

La idea de **shrinkage** (Ledoit & Wolf, 2004) es combinar la covarianza muestral con una **matriz objetivo** más estable:

$$
\hat{\boldsymbol{\Sigma}}_{\text{shrunk}} = (1 - \alpha) \, \hat{\boldsymbol{\Sigma}} + \alpha \, \mathbf{F}
$$

donde:
- $\hat{\boldsymbol{\Sigma}}$: covarianza muestral (alta varianza, bajo sesgo)
- $\mathbf{F}$: matriz objetivo (*target*), típicamente diagonal o proporcional a la identidad (bajo varianza, alto sesgo)
- $\alpha \in [0, 1]$: **coeficiente de contracción** que balancea sesgo y varianza

Cuando $\alpha = 0$, se recupera la muestral. Cuando $\alpha = 1$, se usa solo la matriz objetivo. El valor óptimo de $\alpha$ se determina analíticamente minimizando el error cuadrático medio (Ledoit & Wolf, 2004).

> **Intuición**: la covarianza muestral captura toda la estructura pero con mucho ruido. La contracción "encoge" las covarianzas extremas hacia valores más conservadores, produciendo portafolios más estables.

## 1. Descarga de datos

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import cvxpy as cp
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from sklearn.covariance import ShrunkCovariance, LedoitWolf, EmpiricalCovariance
from sklearn.cluster import KMeans
import scipy.cluster.hierarchy as hac

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams.update({"figure.dpi": 100, "figure.figsize": (12, 5)})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 10)

In [ ]:
tickers = ["AA", "AAPL", "AMZN", "MSFT", "KO", "NVDA", "^GSPC"]
start_date = "2025-01-01"
end_date   = "2025-03-27"

data = yf.download(tickers, start=start_date, end=end_date,
                   auto_adjust=True, progress=False)
closes = data["Close"].dropna()

fig, ax = plt.subplots()
(closes.drop(columns="^GSPC") / closes.drop(columns="^GSPC").iloc[0] * 100).plot(ax=ax)
ax.set_title("Precios normalizados (base 100)")
ax.set_ylabel("Índice")
plt.tight_layout()

---
## 2. Rendimientos y covarianza muestral

In [ ]:
daily_returns = np.log(closes / closes.shift(1)).dropna()
stocks = daily_returns.drop(columns="^GSPC")
stocks.describe()

### Covarianza muestral

La covarianza muestral estándar, que corresponde al estimador de máxima verosimilitud bajo normalidad:

In [ ]:
cov_sample = EmpiricalCovariance().fit(stocks).covariance_

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(pd.DataFrame(cov_sample, index=stocks.columns, columns=stocks.columns),
            annot=True, fmt=".6f", cmap="coolwarm", ax=ax, square=True)
ax.set_title("Matriz de covarianza muestral")
plt.tight_layout()

---
## 3. Estimadores de contracción (Shrinkage)

### 3.1 ShrunkCovariance (sklearn)

`sklearn.covariance.ShrunkCovariance` aplica contracción con un coeficiente $\alpha$ fijo hacia una diagonal:

$$
\hat{\boldsymbol{\Sigma}}_{\text{shrunk}} = (1 - \alpha) \, \hat{\boldsymbol{\Sigma}} + \alpha \, \frac{\text{tr}(\hat{\boldsymbol{\Sigma}})}{n} \, \mathbf{I}_n
$$

donde $\text{tr}(\hat{\boldsymbol{\Sigma}})/n$ es la varianza promedio y $\mathbf{I}_n$ la matriz identidad.

### 3.2 Ledoit-Wolf

`sklearn.covariance.LedoitWolf` estima el coeficiente de contracción $\alpha^*$ **óptimo** analíticamente, minimizando el error cuadrático medio entre el estimador y la covarianza verdadera (Ledoit & Wolf, 2004).

### Comparación

In [ ]:
# Estimadores
shrunk_02 = ShrunkCovariance(shrinkage=0.2).fit(stocks)
shrunk_05 = ShrunkCovariance(shrinkage=0.5).fit(stocks)
lw = LedoitWolf().fit(stocks)

print(f"Coeficiente de contracción Ledoit-Wolf: α* = {lw.shrinkage_:.4f}")

# Comparar matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = [f"Shrunk (α=0.2)", f"Shrunk (α=0.5)", f"Ledoit-Wolf (α={lw.shrinkage_:.3f})"]
covs = [shrunk_02.covariance_, shrunk_05.covariance_, lw.covariance_]

for ax, cov, title in zip(axes, covs, titles):
    sns.heatmap(pd.DataFrame(cov, index=stocks.columns, columns=stocks.columns),
                annot=True, fmt=".6f", cmap="coolwarm", ax=ax, square=True)
    ax.set_title(title)
plt.suptitle("Matrices de covarianza con contracción", fontsize=14)
plt.tight_layout()

---
## 4. Efecto de la contracción en las covarianzas

### Comparación numérica

La contracción reduce las covarianzas off-diagonal (fuera de la diagonal) más que las varianzas (diagonal). Esto estabiliza los portafolios al moderar las relaciones extremas entre activos.

In [ ]:
# Diferencia porcentual entre muestral y Ledoit-Wolf
diff = (lw.covariance_ - cov_sample) / np.abs(cov_sample) * 100
diff_df = pd.DataFrame(diff, index=stocks.columns, columns=stocks.columns)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(diff_df, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Cambio porcentual: Ledoit-Wolf vs. muestral (%)")
plt.tight_layout()

### Efecto en los eigenvalores

La contracción **comprime** la distribución de eigenvalores: los más grandes se reducen y los más pequeños aumentan. Esto mejora el **número de condición** de la matriz, haciéndola más estable numéricamente.

In [ ]:
eig_sample = np.linalg.eigvalsh(cov_sample)
eig_lw = np.linalg.eigvalsh(lw.covariance_)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(eig_sample))
width = 0.35
ax.bar(x - width/2, eig_sample, width, label="Muestral", alpha=0.8)
ax.bar(x + width/2, eig_lw, width, label="Ledoit-Wolf", alpha=0.8)
ax.set_xlabel("Componente")
ax.set_ylabel("Eigenvalor")
ax.set_title("Eigenvalores: covarianza muestral vs. Ledoit-Wolf")
ax.legend()
ax.set_xticks(x)
ax.set_xticklabels([f"λ{i+1}" for i in x])
plt.tight_layout()

cond_sample = np.linalg.cond(cov_sample)
cond_lw = np.linalg.cond(lw.covariance_)
print(f"Número de condición — Muestral: {cond_sample:.1f}, Ledoit-Wolf: {cond_lw:.1f}")

---
## 5. Impacto en la optimización de portafolios

### Frontera eficiente: muestral vs. Ledoit-Wolf

Comparamos las fronteras eficientes obtenidas con la covarianza muestral y con Ledoit-Wolf. La contracción produce fronteras más **conservadoras** y portafolios más **diversificados**.

In [ ]:
def calc_efficient_frontier(returns, Sigma, n_points=100):
    """Frontera eficiente con CVXPY (DCP) usando una covarianza dada."""
    n = returns.shape[1]
    mu = returns.mean().values

    w = cp.Variable(n)
    target_ret = cp.Parameter()
    risk = cp.quad_form(w, Sigma)
    prob = cp.Problem(cp.Minimize(risk),
                      [mu @ w == target_ret, cp.sum(w) == 1, w >= 0])

    means, stds = [], []
    for r in np.linspace(mu.min(), mu.max(), n_points):
        target_ret.value = r
        prob.solve(solver=cp.ECOS, warm_start=True)
        if prob.status == "optimal":
            means.append(r)
            stds.append(np.sqrt(risk.value))
    return means, stds

# Frontera con covarianza muestral
means_s, stds_s = calc_efficient_frontier(stocks, cov_sample)
# Frontera con Ledoit-Wolf
means_lw, stds_lw = calc_efficient_frontier(stocks, lw.covariance_)

fig, ax = plt.subplots(figsize=(10, 7))
ax.plot(stds_s, means_s, "b-", linewidth=2, label="Covarianza muestral")
ax.plot(stds_lw, means_lw, "r--", linewidth=2, label="Ledoit-Wolf")

for col in stocks.columns:
    ax.scatter(stocks[col].std(), stocks[col].mean(), s=80, zorder=5)
    ax.annotate(col, (stocks[col].std(), stocks[col].mean()), fontsize=10)

ax.set_title("Frontera eficiente: covarianza muestral vs. Ledoit-Wolf")
ax.set_xlabel("Riesgo (desviación estándar)")
ax.set_ylabel("Rendimiento esperado")
ax.legend()
plt.tight_layout()

### Comparación de pesos óptimos (máximo Sharpe)

In [ ]:
def max_sharpe_weights(returns, Sigma, risk_free=0.0):
    """Pesos del portafolio de máximo Sharpe con CVXPY (DCP)."""
    n = returns.shape[1]
    mu = returns.mean().values
    excess = mu - risk_free

    y = cp.Variable(n)
    kappa = cp.Variable(nonneg=True)
    prob = cp.Problem(cp.Minimize(cp.quad_form(y, Sigma)),
                      [excess @ y == 1, cp.sum(y) == kappa, y >= 0])
    prob.solve(solver=cp.ECOS)
    return y.value / kappa.value

w_sample = max_sharpe_weights(stocks, cov_sample)
w_lw = max_sharpe_weights(stocks, lw.covariance_)

comparison = pd.DataFrame({
    "Muestral": w_sample,
    "Ledoit-Wolf": w_lw,
    "Diferencia": w_lw - w_sample
}, index=stocks.columns)
comparison

In [ ]:
comparison[["Muestral", "Ledoit-Wolf"]].plot(kind="bar", figsize=(10, 5))
plt.title("Pesos óptimos (max Sharpe): muestral vs. Ledoit-Wolf")
plt.ylabel("Peso")
plt.xticks(rotation=0)
plt.tight_layout()

---
## 6. Selección de activos mediante clustering

Antes de optimizar, es útil **agrupar** activos similares y seleccionar representantes de cada grupo para mejorar la diversificación.

### 6.1 K-Means

Agrupa activos por su perfil rendimiento-riesgo (media y desviación estándar de rendimientos diarios).

In [ ]:
etf_stats = pd.DataFrame({
    "Media": stocks.mean(),
    "Std": stocks.std()
})

n_clusters = 3
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
etf_stats["Cluster"] = kmeans.fit_predict(etf_stats[["Media", "Std"]])

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(etf_stats["Media"], etf_stats["Std"],
                     c=etf_stats["Cluster"], cmap="Set1", s=120, zorder=5)
for ticker, row in etf_stats.iterrows():
    ax.annotate(ticker, (row["Media"], row["Std"]), fontsize=11, ha="left", va="bottom")
ax.set_xlabel("Rendimiento medio diario")
ax.set_ylabel("Desviación estándar diaria")
ax.set_title(f"K-Means clustering ({n_clusters} clusters)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()

### 6.2 Clustering jerárquico

El **dendrograma** muestra la estructura de similitud basada en la correlación de Spearman. Se usa la distancia $d = 1 - \rho$ y el método de Ward.

In [ ]:
corr_spearman = stocks.corr(method="spearman")
Z = hac.linkage(1 - corr_spearman, method="ward")

fig, ax = plt.subplots(figsize=(10, 5))
hac.dendrogram(Z, labels=stocks.columns.tolist(), ax=ax,
               leaf_rotation=0, leaf_font_size=12)
ax.set_title("Dendrograma (correlación de Spearman, método de Ward)")
ax.set_ylabel("Distancia")
plt.tight_layout()

---

## Navegación del curso

← **Anterior**: Clase 4: Ratio de Sharpe  
→ **Siguiente**: Clase 6: Media robusta (Huber)

---

## 7. Referencias bibliográficas

- **Boyd, S. & Vandenberghe, L.** (2004). *Convex Optimization*. Cambridge University Press.
- **Hull, J. C.** (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson. — Cap. 22: Estimating Volatilities and Correlations.
- **Ledoit, O. & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365–411.
- **Ledoit, O. & Wolf, M.** (2004). Honey, I shrunk the sample covariance matrix. *The Journal of Portfolio Management*, 30(4), 110–119.
- **Luenberger, D. G.** (2013). *Investment Science* (2nd ed.). Oxford University Press. — Cap. 6–8.
- **Markowitz, H.** (1952). Portfolio Selection. *The Journal of Finance*, 7(1), 77–91.
- **Michaud, R. O.** (1989). The Markowitz Optimization Enigma: Is 'Optimized' Optimal? *Financial Analysts Journal*, 45(1), 31–42.
- **Tsay, R. S.** (2010). *Analysis of Financial Time Series* (3rd ed.). Wiley.
- **Venegas Martínez, F.** (2008). *Riesgos financieros y económicos* (2a ed.). Cengage Learning.